# V2: feed-forward reconstruction (DUSt3R) -> Gaussian Splatting

Replaces COLMAP's classical SfM with DUSt3R -- a pretrained, generalizing neural
network that predicts camera poses and a *dense* point cloud in one forward
pass, instead of iterative feature matching + bundle adjustment. This is the
first genuinely trained/generalizing model in this project (Category C: used
pretrained, unmodified, inference only).

The output feeds into the exact same `train_gaussian_splatting` from V0 --
that trainer was refactored to be backend-agnostic (`ReconstructedScene`) so
it doesn't care whether the geometry came from COLMAP or DUSt3R.

**Runtime > Change runtime type > GPU (T4 is fine)** before running these cells.

**Honest flag before you run this:** the pose-convention conversion below
(DUSt3R camera-to-world -> gsplat's expected world-to-camera) is written
against DUSt3R's documented API but the exact axis convention has NOT been
verified against a real run yet -- if the trained result looks inverted or
broken, that conversion is the first place to check.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU attached -- go to Runtime > Change runtime type > GPU, then re-run.')

In [ ]:
!git clone https://github.com/yusupildan-wq/Scene-Reconstruction.git
!git clone --recursive https://github.com/naver/dust3r.git
%cd dust3r

In [ ]:
# torch is already installed (with CUDA) by Colab -- do not reinstall it.
!pip install -q -r requirements.txt
# pycolmap isn't used by the DUSt3R path itself, but runner.py imports it at
# module level (needed for the COLMAP path) -- importing anything from runner.py
# runs that line regardless, so it has to be installed here too.
!pip install -q gsplat scipy opencv-python-headless pycolmap

import sys
sys.path.insert(0, '.')  # dust3r package (we're inside the dust3r/ clone)
sys.path.insert(0, '../Scene-Reconstruction/backend')
sys.path.insert(0, '../Scene-Reconstruction/worker')

## Upload your video

Same footage you can reuse from the V0 run if you still have it, or a new one.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_filename = next(iter(uploaded.keys()))
print('Uploaded:', video_filename)

In [ ]:
from pathlib import Path
from app.pipeline import extract_frames

frames_dir = Path('real_video_frames')
result = extract_frames(Path(video_filename), frames_dir)
print(f'{result.total_frames_seen} frames seen, {result.selected_frame_count} selected after blur/redundancy filtering')
frame_paths = sorted(str(p) for p in frames_dir.glob('*.jpg'))

## Run DUSt3R: predict poses + dense point cloud in one forward pass

No iterative bundle adjustment here. Pairing uses a **sliding window** (`swin-3`:
each frame pairs with its ~3 nearest neighbors in the sequence) instead of every
possible pair -- our frames come from a video, so nearby frames overlap heavily
and far-apart frames barely overlap at all; pairing everything with everything
(`scene_graph='complete'`) both wastes compute on uninformative pairs and, at 32
images, produced 992 pairs that exhausted Colab's RAM. `inference` runs the
actual forward passes, and `global_aligner` does a comparatively lightweight
optimization to stitch all the pairwise predictions into one consistent scene --
much cheaper than COLMAP's full bundle adjustment.

In [ ]:
from dust3r.inference import inference
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

device = 'cuda'
model = AsymmetricCroCo3DStereo.from_pretrained(
    'naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt'
).to(device)

images = load_images(frame_paths, size=512)
# swin-3: each frame pairs with ~3 neighbors in sequence order, not every other
# frame -- for 32 images this is on the order of ~100-200 pairs instead of 992,
# and is a better fit for sequential video anyway (see markdown above).
pairs = make_pairs(images, scene_graph='swin-3', prefilter=None, symmetrize=True)
print(f'{len(pairs)} image pairs (was 992 with scene_graph=\"complete\")')
output = inference(pairs, model, device, batch_size=1)

scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
loss = scene.compute_global_alignment(init='mst', niter=300, schedule='cosine', lr=0.01)
print('Global alignment final loss:', loss)

## Convert DUSt3R's output into our ReconstructedScene format

This is the actual bridge between DUSt3R and our existing (already-proven)
Gaussian Splatting trainer -- everything after this cell is identical to the
V0 pipeline, just fed denser, feed-forward-predicted geometry instead of
COLMAP's sparser output.

In [ ]:
import numpy as np

def to_numpy(x):
    return x.detach().cpu().numpy() if hasattr(x, 'detach') else np.asarray(x)

masks = scene.get_masks()
pts3d = scene.get_pts3d()
imgs = scene.imgs
poses = scene.get_im_poses()      # camera-to-world
intrinsics = scene.get_intrinsics()

MAX_POINTS_PER_VIEW = 5000  # DUSt3R gives one point per pixel (~200k+ per view) -- way
# denser than we need or than gsplat can handle at once; subsample after confidence filtering.

all_points, all_colors = [], []
for pts, mask, img in zip(pts3d, masks, imgs):
    pts_np = to_numpy(pts).reshape(-1, 3)
    mask_np = to_numpy(mask).reshape(-1)
    img_np = to_numpy(img).reshape(-1, 3)
    valid_pts = pts_np[mask_np]
    valid_colors = img_np[mask_np]
    if len(valid_pts) > MAX_POINTS_PER_VIEW:
        idx = np.random.choice(len(valid_pts), MAX_POINTS_PER_VIEW, replace=False)
        valid_pts, valid_colors = valid_pts[idx], valid_colors[idx]
    all_points.append(valid_pts)
    all_colors.append(valid_colors)

points_xyz = np.concatenate(all_points).astype(np.float32)
points_rgb = np.concatenate(all_colors).astype(np.float32)
print(f'{points_xyz.shape[0]} points from DUSt3R (COLMAP gave us ~1400-7000 on this same kind of footage)')

camera_viewmats, camera_Ks, camera_images = [], [], []
for i in range(len(imgs)):
    c2w = to_numpy(poses[i])
    # ASSUMPTION flagged in the intro markdown: DUSt3R's camera-to-world matrix
    # is in the same axis convention our gsplat integration already uses
    # successfully (OpenCV/COLMAP-style), so inverting is the only conversion
    # needed. If the trained render looks inverted/broken, revisit this line.
    w2c = np.linalg.inv(c2w).astype(np.float32)
    camera_viewmats.append(w2c)
    camera_Ks.append(to_numpy(intrinsics[i]).astype(np.float32))
    camera_images.append(to_numpy(imgs[i]).astype(np.float32))

print('viewmat shape:', camera_viewmats[0].shape, ' K shape:', camera_Ks[0].shape, ' image shape:', camera_images[0].shape)

In [ ]:
from runner import ReconstructedScene, train_gaussian_splatting

recon_scene = ReconstructedScene(
    points_xyz=points_xyz,
    points_rgb=points_rgb,
    camera_viewmats=camera_viewmats,
    camera_Ks=camera_Ks,
    camera_images=camera_images,
)

# Denser starting point cloud than COLMAP gave us -- may need less aggressive
# densification to reach good coverage. Same iteration budget as the V0 20k run
# for a fair comparison.
gaussians = train_gaussian_splatting(recon_scene, num_iterations=20000, densify_until=16000)
print('Trained', gaussians['means'].shape[0], 'Gaussians')

In [ ]:
import json

export = {
    'means': gaussians['means'].tolist(),
    'quats': gaussians['quats'].tolist(),
    'scales': gaussians['scales'].tolist(),
    'opacities': gaussians['opacities'].tolist(),
    'colors': gaussians['colors'].tolist(),
}
with open('dust3r_scene.json', 'w') as f:
    json.dump(export, f)

files.download('dust3r_scene.json')